In [10]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [12]:
df = pd.read_csv('train.txt' , sep = ';' , header=None , names=['text' , 'emotion'])

In [13]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [14]:
df.isnull().sum()

,0
text,0
emotion,0


In [15]:
# Get unique emotion values
unique_emotions = df['emotion'].unique()

# Create empty dictionary
emotion_numbers = {}

# Counter variable
i = 0

# Assign number to each emotion
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1

# Map emotions to numbers
df['emotion'] = df['emotion'].map(emotion_numbers)

# Display mapping
print(emotion_numbers)

# Preview updated dataframe
df.head()

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [16]:
# Convert text column to lowercase
df['text'] = df['text'].apply(lambda x: x.lower())

#

In [17]:
# 2. Remove all punctuation using str.translate
# We create a translation table mapping punctuation characters to None (deleting them)
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))



In [19]:
df['text'] = df['text'].apply(remove_punc)

In [18]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

In [20]:
df['text'] = df['text'].apply(remove_numbers)

In [21]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new
df['text'] = df['text'].apply(remove_numbers)

In [22]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [23]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [24]:
def remove_stopwords(txt):
    words = word_tokenize(txt)
    cleaned = []
    for word in words:
        if word not in stop_words:
            cleaned.append(word)
    return ' '.join(cleaned) # Corrected return statement

In [25]:
df['text'] = df['text'].apply(remove_stopwords)

In [26]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.33, random_state=42)

In [29]:
X_test

,text
8756,ive made week feel beaten
4660,feel strategy worthwhile
6095,feel worthless weak say want find
304,feel clever nov
8241,im moved ive feeling kind gloomy
...,...
12538,feel really ought assert way smiles pleasant b...
6536,even sure formulate thoughts since put feeling...
13806,really feel quite honoured represent country
7724,ive feeling little overwhelmed whole thing lat...


In [30]:
X_train

,text
8917,feeling quite pleased fact one coupon use grocery
9453,stressed finally come end feel relieved
2829,feel blessed know immense family supporters co...
9681,didnt feel like missed anything
12246,feel listless unable imagine ever working
...,...
13418,love leave reader feeling confused slightly de...
5390,feel delicate
860,starting feel little stressed
15795,feel stressed tired worn shape neglected


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [34]:
bow_vectorizer = CountVectorizer()


In [35]:
# 2. Fit and transform the training data
X_train_bow = bow_vectorizer.fit_transform(X_train)

# 3. Transform the testing data (Only transform, do NOT fit!)
X_test_bow = bow_vectorizer.transform(X_test)

In [36]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [37]:
# 1. Initialize the Multinomial Naive Bayes model
nb = MultinomialNB()

# 2. Train (fit) the model on the training features and labels
nb.fit(X_train_bow, y_train)

# 3. Predict the emotions for the test set
y_pred = nb.predict(X_test_bow)

# 4. Calculate and print the model's accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7650


In [38]:
# 1. Initialize the TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# 2. Fit and transform the training text data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# 3. Transform the testing text data (Only transform!)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 4. Initialize and train the Naive Bayes model
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

# 5. Make predictions and print the accuracy
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred_tfidf)

print(f"TF-IDF Model Accuracy: {accuracy:.4f}")

TF-IDF Model Accuracy: 0.6610


In [39]:
from sklearn.linear_model import LogisticRegression

In [40]:
# 1. Initialize the TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# 2. Fit and transform the training text data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# 3. Transform the testing text data (Only transform!)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 4. Initialize and train the Logistic Regression model
# (Note: max_iter is increased to 1000 to ensure the solver converges on text datasets)
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# 5. Make predictions on the test set
y_pred_lr = lr_model.predict(X_test_tfidf)

# 6. Calculate and print the final accuracy
accuracy = accuracy_score(y_test, y_pred_lr)
print(f"Logistic Regression Model Accuracy: {accuracy:.4f}")

Logistic Regression Model Accuracy: 0.8473
